# Can a team's record predict tonight's winner?
Our DS 150 dataset has two rows per game, one per team. We'll make one pick per game using each team's record at tipoff. Run the cells in order from the repository root. Only pandas and the included CSV are needed.

**Prediction rule:** Pick the team with the higher win percentage so far this season. Pick the home team when records tie. Give a team a starting percentage of 0.500 before its first game. Compare the rule with always picking the home team on the same 2025–26 games. Both rules are fixed in advance and require no training.

In [1]:
import pandas as pd

teams = pd.read_csv("data/all_games_clean.csv", parse_dates=["GAME_DATE"])
teams = teams.sort_values(["SEASON", "TEAM_ID", "GAME_DATE"])
assert not teams.duplicated(["SEASON", "TEAM_ID", "GAME_DATE"]).any()
assert teams["WIN"].isin([0, 1]).all()
assert teams["HOME"].isin([0, 1]).all()
print(f"{len(teams):,} team rows across {teams['GAME_ID'].nunique():,} games.")

12,300 team rows across 6,150 games.


## What did we know at tipoff?
For a team entering its fourth game after two wins and a loss, its record at tipoff is 2–1 regardless of tonight's outcome. Counting tonight's win would leak the answer into the prediction. Group by season as well as team so records reset each fall.

In [2]:
by_team = teams.groupby(["SEASON", "TEAM_ID"])
teams["games_before"] = by_team.cumcount()
teams["wins_before"] = by_team["WIN"].cumsum() - teams["WIN"]
played = teams["games_before"].replace(0, float("nan"))
teams["pregame_pct"] = (teams["wins_before"] / played).fillna(0.5)

wolves = teams.loc[(teams["TEAM_ABBREVIATION"] == "MIN") & (teams["SEASON"] == "2025-26")]
print(wolves[["GAME_DATE", "MATCHUP", "games_before", "wins_before", "pregame_pct", "WL"]].head().to_string(index=False))

 GAME_DATE     MATCHUP  games_before  wins_before  pregame_pct WL
2025-10-22   MIN @ POR             0            0     0.500000  W
2025-10-24   MIN @ LAL             1            1     1.000000  L
2025-10-26 MIN vs. IND             2            1     0.500000  W
2025-10-27 MIN vs. DEN             3            2     0.666667  L
2025-10-29 MIN vs. LAL             4            2     0.500000  L


## Put both teams on one row
Some games have two away rows in the CSV. They still count toward the teams' records. Exclude them from scoring because the home-team rule needs a designated home team.

In [3]:
by_game = teams.groupby("GAME_ID")
assert by_game.size().eq(2).all()
assert by_game["WIN"].sum().eq(1).all()
eligible = teams.loc[by_game["HOME"].transform("sum") == 1]
home = eligible.loc[eligible["HOME"] == 1, ["GAME_ID", "SEASON", "GAME_DATE", "TEAM_ABBREVIATION", "pregame_pct", "WIN"]]
away = eligible.loc[eligible["HOME"] == 0, ["GAME_ID", "TEAM_ABBREVIATION", "pregame_pct"]]
games = home.merge(away, on="GAME_ID", suffixes=("_home", "_away"), validate="one_to_one")
test = games.loc[games["SEASON"] == "2025-26"].copy()
season_games = teams.loc[teams["SEASON"] == "2025-26", "GAME_ID"].nunique()
print(f"Scoring {len(test):,} of {season_games:,} games in 2025–26.")

Scoring 1,225 of 1,230 games in 2025–26.


## Score the picks
A home pick is correct when the home team wins. An away pick is correct when the home team loses. The two rules use exactly the same set of games.

In [4]:
pick_home = test["pregame_pct_home"] >= test["pregame_pct_away"]
home_won = test["WIN"] == 1
home_correct = home_won.sum()
record_correct = (pick_home == home_won).sum()

for label, correct in [("Always home", home_correct), ("Better pregame record", record_correct)]:
    print(f"{label}: {correct:,}/{len(test):,} correct ({correct / len(test):.1%})")
print(f"Extra correct picks: {record_correct - home_correct:+d}")

Always home: 679/1,225 correct (55.4%)
Better pregame record: 821/1,225 correct (67.0%)
Extra correct picks: +142


## Your turn
Among games where the rule picked the **away team**, how often was it right? Start by selecting those rows, then count the away wins. Compare your result with always picking home on that same subset. Explain in one sentence how these games account for the entire difference between the two rules.

**Next experiment:** Fit a PyTorch logistic regression on the first four seasons to predict the probability of a home win from the difference in pregame win percentages. Choose model settings using earlier seasons. Report its 2025–26 accuracy alongside both rules above. Since we've already seen this season's baseline results, save a future season for a fresh evaluation if you start tuning to these results.